# Interrogatorio
Diego Torres Durante

## El Reto y Contexto: 
Siguen trabajando para la aseguradora marítima. Ya saben cómo obtener la radiografía general (.describe()), pero el director de riesgos necesita respuestas segmentadas y sospecha que hubo fraudes en las tarifas cobradas. Deben usar técnicas avanzadas de sumarización en memoria para confirmarlo.

## Instrucciones Técnicas: 
Abran un Jupyter Notebook nuevo carguen el dataset crudo directamente desde el repositorio:

In [1]:
import pandas as pd
 
 
url = "../datasets/titanic.csv"
df = pd.read_csv(url)

In [2]:
df.sample(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
538,539,0,3,"Risien, Mr. Samuel Beard",male,NaN,0,0,364498,14.5000,NaN,S
701,702,1,1,"Silverthorne, Mr. Spencer Victor",male,35.0,0,0,PC 17475,26.2875,E24,S
576,577,1,2,"Garside, Miss. Ethel",female,34.0,0,0,243880,13.0000,NaN,S
610,611,0,3,"Andersson, Mrs. Anders Johan (Alfrida Konstant...",female,39.0,1,5,347082,31.2750,NaN,S
386,387,0,3,"Goodwin, Master. Sidney Leonard",male,1.0,5,2,CA 2144,46.9000,NaN,S
423,424,0,3,"Danbom, Mrs. Ernst Gilbert (Anna Sigrid Maria ...",female,28.0,1,1,347080,14.4000,NaN,S
746,747,0,3,"Abbott, Mr. Rossmore Edward",male,16.0,1,1,C.A. 2673,20.2500,NaN,S
737,738,1,1,"Lesurer, Mr. Gustave J",male,35.0,0,0,PC 17755,512.3292,B101,C
377,378,0,1,"Widener, Mr. Harry Elkins",male,27.0,0,2,113503,211.5000,C82,C
623,624,0,3,"Hansen, Mr. Henry Damsgaard",male,21.0,0,0,350029,7.8542,NaN,S


## Preguntas de Análisis 
(Para responder con código y celdas Markdown):

### Pregunta A (Sumarización Categórica): 
Usen la función .value_counts(normalize=True) en la columna Survived (0 = Murió, 1 = Sobrevivió). ¿Cuál es la tasa de supervivencia global del barco expresada en porcentaje?


In [3]:
df["Survived"].value_counts(normalize=True)


Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

#### Respuesta:
Sobrevivió el 38.38%
Murió el 61.61%

---------

### Pregunta B (Agrupación y Agregación): 
El famoso código marítimo era "mujeres y niños primero". Comprobemos esto matemáticamente. Ejecuten una agrupación por la columna de género y calculen el promedio de la columna Survived. ¿Qué porcentaje exacto de mujeres sobrevivió en contraste con los hombres?

In [4]:
count = df.groupby("Sex")["Survived"].sum()
print("asd", count)

total = df["PassengerId"].count()
print("total", total)

per_female = count.female/total * 100
per_male = count.male/total *100

print("percentage female: ", per_female)
print("percentage male: ", per_male)



asd Sex
female    233
male      109
Name: Survived, dtype: int64
total 891
percentage female:  26.15039281705948
percentage male:  12.2334455667789


#### Respuesta:
De toda la población, 26.15% fueron mujeres que sobrevivieron, mientras que el 12.23% fueron hombres que sobrevivieron.

----

### Pregunta C: 
El director financiero nota que la tarifa máxima (Fare) cobrada fue de más de 500 libras, mientras que el promedio ronda las 32. Sospecha de outliers. Utilicen las herramientas de dispersión en Pandas para confirmarlo:

a. Calculen el Cuartil 1 (25%) y el Cuartil 3 (75%) de la columna Fare usando el método .quantile([0.25, 0.75]).

b. Calculen matemáticamente el Rango Intercuartílico (IQR = Q3 - Q1).

c. Calculen el límite superior aceptable (Q3 + 1.5 * IQR). Escriban una línea de código para filtrar el DataFrame y descubrir exactamente cuántos pasajeros pagaron una tarifa por encima de ese límite matemático. ¿A qué clase (Pclass) pertenecían la mayoría de ellos?

In [5]:
Q1 = df["Fare"].quantile(0.25)
Q3 = df["Fare"].quantile(0.75)

IQR = Q3-Q1

MIN = Q1 - IQR*1.5
MAX = Q3 + IQR*1.5


filtered = df[df["Fare"]>=MAX]
print("Limite superior", MAX)
filtered.groupby("Pclass")["Fare"].count()
# df.groupby("Pclass").filter(df["Fare"]>MAX)

# limit.groupby("Pclass")


Limite superior 65.6344


Pclass
1    104
2      5
3      7
Name: Fare, dtype: int64

#### Respuesta:
1ra clase, 104 pasajeros.

-----

### Pregunta D: 
Calculen la media y la mediana de la columna Fare. Notarán una diferencia enorme entre ambos valores. Matemáticamente, ¿qué significa que la media sea tan superior a la mediana? Si en el futuro utilizamos un algoritmo basado en distancias euclidianas (como K-Nearest Neighbors) sin escalar previamente esta variable, ¿cómo afectará esta asimetría al aprendizaje del modelo?

In [6]:
print(df["Fare"].mean())
print(df["Fare"].median())



32.204207968574636
14.4542


La media es mayor por el sesgo a la derecha de los outliers. la asimetría haría que el aprendizaje de un model tenga un sesgo.


### Pregunta E: 
En la semana 2 vimos que el muestreo aleatorio simple es peligroso. El objetivo es entrenar un modelo que prediga la supervivencia (Survived), pero las clases están desbalanceadas. Escriban el código en Pandas para extraer una muestra de exactamente 150 pasajeros garantizando que la proporción de sobrevivientes y no sobrevivientes en la muestra sea idéntica a la de la base de datos completa. ¿Qué sesgo evitan al hacer esto?


In [7]:
print(df["Survived"].value_counts())

sample_size = 150

total = len(df)
survived = df["Survived"].sum()

n_survived = round(sample_size * survived / total)
n_not_survived = sample_size - n_survived

print(n_survived, n_not_survived,"\n\n")

sample = pd.concat([
    df[df["Survived"] == 1].sample(n=n_survived, random_state=42),
    df[df["Survived"] == 0].sample(n=n_not_survived, random_state=42)
])

print(sample["Survived"].value_counts())


Survived
0    549
1    342
Name: count, dtype: int64
58 92 


Survived
0    92
1    58
Name: count, dtype: int64


si no se mantienen las proporciones se introduce un sesgo de tipo sampling bias, y no representaría adecuadamente a la población.


### Pregunta F: 
Si ejecutan df.groupby('Survived')['Age'].mean(), Pandas calculará el promedio de edad de los que vivieron y los que murieron. Sin embargo, por defecto, Pandas ignora los valores NaN al calcular la media. Si resulta que la gran mayoría de las edades faltantes (NaN) pertenecían a pasajeros de 3ra clase que murieron, ¿qué sesgo estadístico estamos introduciendo involuntariamente en el resultado de esa función y cómo afectaría la inferencia de nuestro modelo?

In [8]:
df.groupby("Survived")["Age"].mean()

Survived
0    30.626179
1    28.343690
Name: Age, dtype: float64

Depende la cantidad de datos faltantes el sesgo que puede ocasionar, al final el modelo puede aprender una relación incorrecta entre la edad, clase social y supervivencia que no representa la realidad.

### Pregunta G: 
Con el cálculo del IQR en la Pregunta C, determinaron que los boletos de 512 libras son outliers matemáticos. En Machine Learning, los valores atípicos pueden representar errores de captura (ruido que aumenta el error irreducible) o casos especiales válidos (señal). Investigando la naturaleza de un barco de lujo, ¿deberíamos eliminar estas filas con .drop() antes de entrenar nuestro modelo? Justifiquen su respuesta arquitectónica.


No deberian eliminarse, se puede usar una escala logaritmica o similar para poder considerar los outliers, ya que hasta cierto punto son datos utiles, nos dan información, el tema es saber usarlos sin sesgar.

### Pregunta H: 
Observen la columna Name. Es texto libre (dato no estructurado), pero contiene títulos ocultos como "Mr.", "Mrs.", "Miss." o "Master.". Si lograran extraer ese título usando expresiones regulares en Pandas, podrían hacer un .groupby('Titulo')['Age'].median(). ¿Por qué imputar las edades faltantes basándose en la mediana del "Título" (ej. "Master" = niño, "Mr" = adulto) sería estadísticamente superior y reduciría el error de nuestro futuro modelo, en comparación con usar la mediana global?

Es mejor usar estratos para imputar edades porque estos nos dan información de cierto rango valido que estos deberían tener en caso de que sus edades hayan sido escritas. Usar una media global permitiría iputar una edad que lógicamente no queda dentro del rango que se puede inferir con base al título del tipo de persona.

### Pregunta I: 
Calculen la varianza de la columna Survived. Dado que es una variable categórica codificada como 0 y 1, el resultado numérico estará cerca de $0.23$. Matemáticamente, ¿qué significaría si la varianza de esta variable fuera exactamente 0.0? ¿Qué pasaría si intentan entrenar un algoritmo de clasificación con un dataset donde la variable de respuesta tiene varianza 0.0?


In [9]:
print(df["Survived"].var())
print(df["Survived"].std())


0.2367722165474984
0.4865924542648585


si la varianza es 0, o todos vivieron o todos murieron, no hay variación entre los datos, son el mismo dato. el agoritmo de clasificación solo clasificaría con dicha clasificación, si se esperan más de ua categoría no funcionaría.

### Pregunta J: 
Ejecuten una agrupación por tres niveles al mismo tiempo y cuenten cuántos pasajeros hay en cada subgrupo: df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count(). Notarán que algunos subgrupos tienen 1 o 2 pasajeros. Si un algoritmo intenta extraer reglas de probabilidad de grupos tan pequeños, se enfrentará a la "Maldición de la Dimensionalidad" (Curse of Dimensionality). ¿Qué fenómeno perjudicial (sobreajuste o subajuste) ocurrirá inevitablemente si dejamos que el modelo aprenda reglas basadas en esos grupos de 1 solo pasajero?


In [10]:
df.groupby(['Pclass', 'Sex', 'Embarked'])['PassengerId'].count()

Pclass  Sex     Embarked
1       female  C            43
                Q             1
                S            48
        male    C            42
                Q             1
                S            79
2       female  C             7
                Q             2
                S            67
        male    C            10
                Q             1
                S            97
3       female  C            23
                Q            33
                S            88
        male    C            43
                Q            39
                S           265
Name: PassengerId, dtype: int64

habría un sobre ajuste, que si luego el algoritmo se aplica con otro dataset, el rendimiento no sería bueno por que aprendió a clasificar en base al dataset de entrenamiento y no aprendió las variables claves para determinar las clasificaciones que hacen sentido.